<a href="https://colab.research.google.com/github/Mc-cloud/chessRL/blob/main/agents/Agent_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 56.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=16a1866a9d64a242fba8f41138c7b98b54224c625da98ec079266ab38d404d6b
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [3]:
import math
import random
import chess
import copy

class Node:
  def __init__(self, state : chess.Board, parent = None, prior_prob = 0.0):
    self.state = state
    self.parent = parent
    self.children = {}

    self.n_visits = 0
    self.value_sum = 0
    self.q_value = 0
    self.prior_prob = prior_prob

  def expand(self, action_probs):
    for move, prob in action_probs.items():
      if move not in self.children:
        next_state = self.state.copy()
        next_state.push(chess.Move.from_uci(move))

        self.children[move] = Node(state=next_state, parent= self, prior_prob = prob)

def is_expended(self):
  return len(self.children) > 0

def best_child(self,c):
  best_score = -math.inf
  best_action = None
  best_child = None

  for action, child in self.children.items():
    q_val = child.q_value
    u_val = c * child.prior_prob * math.sqrt(self.n_visits)/ (1 + child.n_visits)
    puct_score = q_val + u_val

    if puct_score > best_score :
      best_score = puct_score
      best_action = action
      best_child = child

    return best_action, best_child

  def backpropagate(self, value):
    self.n_visits += 1
    self.value_sum += value
    self.q_value = self.value_sum / self.n_visits

    if self.parent is not None:
      self.parent.backpropagate(-value)

In [4]:
class MCTS:
  def __init__(self, neural_net, c = 1.5, n_simulations = 800):
    self.nn = neural_net
    self.c = c
    self.n_simulations = n_simulations

  def search(self, initial_state : chess.Board):
    root = Node(state = initial_state)

    for _ in range(self.n_simulations):
      node = root

      while node.is_expended():
        action, node = node.best_child(self.c)

      if node.state.is_game_over():
        value = -1.0 if node.state.is_checkmate() else 0.0
      else :
        action_probs, value = self.nn.predict(node.state)

        legal_moves = [m.uci() for m in node.state.legal_moves]
        legal_probs = {m : prob for m, prob in action_probs.items() if m in legal_moves}

        sum_probs = sum(legal_probs.values())

        if sum_probs > 0:
          legal_probs = {m: prob / sum_probs for m, prob in legal_probs.items()}
        else :
          legal_probs = {m : 1.0/len(legal_moves) for m in legal_moves}

        node.expand(legal_probs)

      node.backpropagate(-value)

    action_visits = {action : child.n_visits for action, child in root.children.items()}
    sum_visits = sum(action_visits.values())

    mcts_policy = {action : visits/sum_visits for action, visits in action_visits.items()}

    return mcts_policy